# `ShellToolMiddleware`

Middleware that registers a persistent shell tool for an agent.

The middleware creates one long-lived shell session for each agent run. Commands execute sequentially inside the same session, so working-directory changes, environment changes, and files created by earlier commands remain available to later commands.

The shell is started before the agent runs and cleaned up after the agent finishes.

- Bases: `AgentMiddleware[ShellToolState[ResponseT], ContextT, ResponseT]`
- State schema: `ShellToolState`

> **Security warning**
>
> The default execution policy is `HostExecutionPolicy`, which gives commands access
> to the host environment. Use an isolated container, virtual machine, Codex sandbox,
> or Docker execution policy when commands are not fully trusted.
>
> Output redaction occurs **after** a command executes. It does not prevent a command
> from reading or transmitting sensitive information.

## Constructor

```python
ShellToolMiddleware(
    workspace_root: str | Path | None = None, # Persistent working directory
    *,
    startup_commands: tuple[str, ...] | list[str] | str | None = None,
    shutdown_commands: tuple[str, ...] | list[str] | str | None = None,
    execution_policy: BaseExecutionPolicy | None = None,
    redaction_rules: tuple[RedactionRule, ...] | list[RedactionRule] | None = None,
    tool_description: str | None = None,
    tool_name: str = "shell",
    shell_command: Sequence[str] | str | None = None,
    env: Mapping[str, Any] | None = None
)
```

## Parameters

* `workspace_root` — Base directory used by the shell session.
  * Accepts a string or `Path`.
  * When omitted, a temporary directory is created when the agent starts.
  * The temporary directory is removed when the agent ends.
  * The directory is created automatically when it does not exist.

* `startup_commands` — Command or commands executed sequentially after the shell starts.
  * Accepts one string, a list of strings, a tuple of strings, or `None`.
  * A failed or timed-out startup command prevents the shell session from starting.
  * Startup commands also run after an explicit shell restart.

* `shutdown_commands` — Command or commands executed before the shell session is released.
  * Accepts one string, a list of strings, a tuple of strings, or `None`.
  * Failures and timeouts are logged rather than raised, allowing cleanup to continue.

* `execution_policy` — Policy controlling how the subprocess is created and managed.
  * Controls command, startup, and termination timeouts.
  * Controls output line and byte limits.
  * Defaults to `HostExecutionPolicy()`.

* `redaction_rules` — Rules applied to command output before it is returned to the model.
  * Each `RedactionRule` is resolved during middleware construction.
  * Rules may redact, mask, hash, or block matching output depending on their strategy.
  * Matches are included in the returned tool artifact.

* `tool_description` — Custom description for the registered shell tool.
  * Uses `DEFAULT_TOOL_DESCRIPTION` when omitted.

* `tool_name` — Name assigned to the registered tool.
  * Default: `"shell"`

* `shell_command` — Shell executable and optional arguments.
  * A string becomes a one-element command tuple.
  * A sequence is converted to a tuple.
  * Default: `("/bin/bash",)`
  * An empty sequence raises `ValueError`.

* `env` — Environment variables supplied to the shell process.
  * Keys must be strings.
  * Values are converted to strings.
  * When omitted, no explicit environment mapping is configured by the middleware.

## Attributes

* `state_schema` — Set to `ShellToolState`.
* `tools` — One-element list containing the registered shell tool.
* `_workspace_root` — Configured workspace path or `None`.
* `_tool_name` — Registered shell-tool name.
* `_shell_command` — Normalized shell command tuple.
* `_environment` — Normalized environment dictionary or `None`.
* `_execution_policy` — Selected execution policy.
* `_redaction_rules` — Tuple of resolved redaction rules.
* `_startup_commands` — Startup commands normalized to a tuple.
* `_shutdown_commands` — Shutdown commands normalized to a tuple.
* `_shell_tool` — LangChain tool created by the constructor.

## Registered Tool

The middleware registers a tool whose default name is:

```text
shell
```

### Input

```python
{
    "command": str | None,
    "restart": bool
}
```

Exactly one operation must be requested:

```python
{"command": "pwd && ls -la"}
```

or:

```python
{"restart": True}
```

Providing neither operation, or providing both operations together, fails input validation.

### Default Description

The default tool description instructs the agent to:

* Confirm the working directory with commands such as `pwd` or `ls`.
* Ensure parent directories exist.
* Prefer absolute paths.
* Quote paths containing spaces.
* Chain commands with `&&` or `;` instead of embedding newlines.
* Avoid unnecessary `cd` commands.
* Expect large outputs to be truncated.
* Expect long-running commands to be terminated after the configured timeout.

## Methods

1. `before_agent`: Creates or reuses the persistent shell resources before the agent starts.
   * Starts the subprocess when resources do not already exist.
   * Runs all configured startup commands.
   * Stores the resources in the agent state.
   * Reuses existing resources after an interrupt or resume.
   - **Syntax:**
     ```python
     before_agent(
         self,
         state: ShellToolState[ResponseT], # Current agent state
         runtime: Runtime[ContextT] # Runtime context
     ) -> dict[str, Any] | None
     ```

2. `abefore_agent`: Asynchronous version of `before_agent`.
   * Runs the synchronous setup through `run_in_executor`.
   - **Syntax:**
     ```python
     async def abefore_agent(
         self,
         state: ShellToolState[ResponseT],
         runtime: Runtime[ContextT]
     ) -> dict[str, Any] | None
     ```

3. `after_agent`: Runs shutdown commands and releases shell resources.
   * Does nothing when the shell resources were never created.
   * Always invokes the resource finalizer, even when shutdown-command execution fails.
   - **Syntax:**
     ```python
     after_agent(
         self,
         state: ShellToolState[ResponseT],
         runtime: Runtime[ContextT]
     ) -> None
     ```

4. `aafter_agent`: Asynchronous lifecycle hook for cleanup.
   * Delegates directly to `after_agent`.
   - **Syntax:**
     ```python
     async def aafter_agent(
         self,
         state: ShellToolState[ResponseT],
         runtime: Runtime[ContextT]
     ) -> None
     ```

5. `_normalize_commands`: Converts optional command configuration into a tuple.
   - **Syntax:**
     ```python
     _normalize_commands(
         commands: tuple[str, ...] | list[str] | str | None
     ) -> tuple[str, ...]
     ```

   Behaviour:

   ```text
   None          -> ()
   "pwd"         -> ("pwd",)
   ["pwd", "ls"] -> ("pwd", "ls")
   ```

6. `_normalize_shell_command`: Converts the configured shell command into a tuple.
   * Returns `("/bin/bash",)` when no command is supplied.
   * Raises `ValueError` for an empty sequence.
   - **Syntax:**
     ```python
     _normalize_shell_command(
         shell_command: Sequence[str] | str | None
     ) -> tuple[str, ...]
     ```

7. `_normalize_env`: Validates and stringifies environment variables.
   * Returns `None` when no mapping is supplied.
   * Raises `TypeError` when an environment-variable name is not a string.
   - **Syntax:**
     ```python
     _normalize_env(
         env: Mapping[str, Any] | None
     ) -> dict[str, str] | None
     ```

8. `_get_or_create_resources`: Returns existing session resources or creates them.
   * Supports resumability by preserving live resources in private agent state.
   - **Syntax:**
     ```python
     _get_or_create_resources(
         self,
         state: ShellToolState[ResponseT]
     ) -> _SessionResources
     ```

9. `_create_resources`: Creates the workspace and starts a new `ShellSession`.
   * Creates a temporary directory when `workspace_root` is absent.
   * Cleans up the session and temporary directory when startup fails.
   - **Syntax:**
     ```python
     _create_resources(
         self
     ) -> _SessionResources
     ```

10. `_run_startup_commands`: Runs startup commands sequentially.
    * Uses `execution_policy.startup_timeout`.
    * Raises `RuntimeError` when a command times out or exits unsuccessfully.
    - **Syntax:**
      ```python
      _run_startup_commands(
          self,
          session: ShellSession
      ) -> None
      ```

11. `_run_shutdown_commands`: Runs shutdown commands sequentially.
    * Uses `execution_policy.command_timeout`.
    * Logs timeouts, non-zero exit codes, and execution failures.
    * Does not stop remaining cleanup.
    - **Syntax:**
      ```python
      _run_shutdown_commands(
          self,
          session: ShellSession
      ) -> None
      ```

12. `_apply_redactions`: Applies every resolved redaction rule to command output.
    * Returns the transformed content.
    * Groups detected matches by PII type.
    * May raise `PIIDetectionError` when a rule uses the block strategy.
    - **Syntax:**
      ```python
      _apply_redactions(
          self,
          content: str
      ) -> tuple[str, dict[str, list[PIIMatch]]]
      ```

13. `_run_shell_tool`: Executes a command or restarts the session.
    * Returns `ToolMessage` when a tool-call ID is present.
    * Returns plain text when called without a tool-call ID.
    * Converts command timeouts into error results.
    * Applies redaction before returning output.
    * Appends truncation notices and non-zero exit codes.
    - **Syntax:**
      ```python
      _run_shell_tool(
          self,
          resources: _SessionResources,
          payload: dict[str, Any],
          *,
          tool_call_id: str | None
      ) -> ToolMessage | str
      ```

14. `_format_tool_message`: Formats shell output for tool execution.
    * Returns plain content when `tool_call_id=None`.
    * Otherwise returns a `ToolMessage` containing status and artifact metadata.
    - **Syntax:**
      ```python
      _format_tool_message(
          self,
          content: str,
          tool_call_id: str | None,
          *,
          status: Literal["success", "error"],
          artifact: dict[str, Any] | None = None
      ) -> ToolMessage | str
      ```

# `ShellToolState`

Agent-state extension used to retain shell resources for the duration of an agent run.

- Bases: `AgentState[ResponseT]`

## Field

```python
shell_session_resources: NotRequired[
    Annotated[
        _SessionResources | None,
        UntrackedValue,
        PrivateStateAttr
    ]
]
```

The value is:

* Optional.
* Private to middleware state.
* Untracked by normal LangGraph channel persistence.
* Reused when an interrupted run resumes in the same process.

# `CommandExecutionResult`

Immutable structured result returned by `ShellSession.execute`.

```python
@dataclass(frozen=True)
class CommandExecutionResult:
    output: str
    exit_code: int | None
    timed_out: bool
    truncated_by_lines: bool
    truncated_by_bytes: bool
    total_lines: int
    total_bytes: int
```

## Attributes

* `output` — Captured standard output and formatted standard error.
* `exit_code` — Shell exit status, or `None` when unavailable.
* `timed_out` — Whether execution exceeded the command timeout.
* `truncated_by_lines` — Whether output exceeded the configured line limit.
* `truncated_by_bytes` — Whether output exceeded the configured byte limit.
* `total_lines` — Total number of output lines observed, including discarded lines.
* `total_bytes` — Total output bytes observed, including discarded output.

# `ShellSession`

Persistent subprocess wrapper that executes sequential commands in the same shell.

## Constructor

```python
ShellSession(
    workspace: Path,
    policy: BaseExecutionPolicy,
    command: tuple[str, ...],
    environment: Mapping[str, str]
)
```

## Attributes

* `_workspace` — Working directory passed to the execution policy.
* `_policy` — Policy used to spawn and control the subprocess.
* `_command` — Shell executable and argument tuple.
* `_environment` — Environment supplied to the shell.
* `_process` — Active `subprocess.Popen` object or `None`.
* `_stdin` — Standard-input pipe for sending commands.
* `_queue` — Thread-safe queue receiving stdout and stderr lines.
* `_lock` — Prevents overlapping command execution.
* `_stdout_thread` — Background stdout-reader thread.
* `_stderr_thread` — Background stderr-reader thread.
* `_terminated` — Indicates whether shutdown has already occurred.

## Methods

1. `start`: Starts the shell subprocess and output-reader threads.
   * Returns immediately when a live process already exists.
   * Raises `RuntimeError` when stdin, stdout, or stderr pipes are unavailable.
   - **Syntax:**
     ```python
     start(
         self
     ) -> None
     ```

2. `restart`: Stops and starts the shell process.
   - **Syntax:**
     ```python
     restart(
         self
     ) -> None
     ```

3. `stop`: Gracefully exits the shell and force-kills it when necessary.
   * Writes `exit` to stdin.
   * Waits for the supplied timeout.
   * Kills the process or dedicated process group when graceful termination fails.
   - **Syntax:**
     ```python
     stop(
         self,
         timeout: float
     ) -> None
     ```

4. `execute`: Runs one command in the persistent shell.
   * Serializes execution with a thread lock.
   * Adds a unique completion marker followed by `$?` to obtain the exit code.
   * Captures stdout and stderr through background threads.
   * Restarts the shell after a timeout.
   * Handles commands such as `exit` that terminate the shell before the marker is written.
   - **Syntax:**
     ```python
     execute(
         self,
         command: str,
         *,
         timeout: float
     ) -> CommandExecutionResult
     ```

## Output Collection

Standard error is included in the returned text with a prefix:

```text
[stderr] <message>
```

A unique marker is printed after every command:

```text
__LC_SHELL_DONE__<uuid> <exit-code>
```

The marker is consumed internally and is not included in normal command output.

The session counts all observed lines and bytes, even after the configured output limit has been exceeded. Excess content is discarded while the counters continue increasing.

## Timeout Behaviour

When a command exceeds its timeout:

1. The command is marked as timed out.
2. The current shell is restarted.
3. The returned result contains:
   ```python
   {
       "output": "",
       "exit_code": None,
       "timed_out": True,
   }
   ```
4. The tool layer returns an error message containing the configured timeout.

Because the shell is restarted, state created only inside the terminated shell process—such as shell-local variables or the current directory—is reset. Files written to the workspace remain available.

# `_ShellToolInput`

Internal Pydantic model used as the registered tool's input schema.

```python
class _ShellToolInput(BaseModel):
    command: str | None = None
    restart: bool | None = None
    runtime: Annotated[Any, SkipJsonSchema()] = None
```

## Validation

```python
validate_payload(
    self
) -> _ShellToolInput
```

Validation fails when:

```text
command is None and restart is not true
```

or:

```text
command is supplied and restart is true
```

The `runtime` field is excluded from the generated JSON schema and is used as a workaround for injected `ToolRuntime`.

# Tool Result Format

Successful agent tool calls return:

```python
ToolMessage(
    content="<command output>",
    tool_call_id="<tool-call-id>",
    name="<configured tool name>",
    status="success",
    artifact={
        "timed_out": False,
        "exit_code": 0,
        "truncated_by_lines": False,
        "truncated_by_bytes": False,
        "total_lines": 1,
        "total_bytes": 12,
        "redaction_matches": {},
    },
)
```

A non-zero exit code changes the status to `"error"` and appends the exit code to the content:

```text
<command output>

Exit code: 1
```

An empty successful output is represented as:

```text
<no output>
```

## Timeout Artifact

```python
{
    "timed_out": True,
    "exit_code": None,
}
```

## Blocked-Output Artifact

When a redaction rule blocks detected content:

```python
{
    "timed_out": False,
    "exit_code": <exit code>,
    "matches": {
        "<pii type>": [<PIIMatch>, ...]
    },
}
```

# Output Truncation

Output may be truncated independently by:

* Maximum output lines.
* Maximum output bytes.

The returned content includes a notice such as:

```text
... Output truncated at 1000 lines (observed 1480).
```

or:

```text
... Output truncated at 100000 bytes (observed 137642).
```

The exact limits come from the selected execution policy.

# Execution Policies

This module re-exports the following execution policies from
`langchain.agents.middleware._execution`.

## `HostExecutionPolicy`

Runs the shell directly on the host.

Use it only when the surrounding process already runs inside a trusted isolation boundary or when all commands are trusted.

## `CodexSandboxExecutionPolicy`

Uses the Codex CLI sandbox to add filesystem and syscall restrictions when the CLI is available.

## `DockerExecutionPolicy`

Runs the shell in a separate Docker container for each agent run.

It can provide stronger isolation, including container-level filesystem controls and user remapping.

The execution-policy classes define values used by this middleware, including:

```text
command_timeout
startup_timeout
termination_timeout
max_output_lines
max_output_bytes
```

# `RedactionRule`

This module re-exports `RedactionRule` from
`langchain.agents.middleware._redaction`.

Rules are resolved during middleware construction and applied to command output after execution.

> Redaction protects what is returned to the model. It does not stop the command
> itself from accessing sensitive data.

# Examples

## Basic Persistent Shell

```python
from langchain.agents import create_agent
from langchain.agents.middleware import ShellToolMiddleware

agent = create_agent(
    model="openai:gpt-5.5",
    middleware=[
        ShellToolMiddleware(
            workspace_root="./workspace"
        )
    ],
)
```

Commands issued during the run share the same shell session and workspace.

## Temporary Workspace

```python
middleware = ShellToolMiddleware()
```

A temporary workspace is created when the agent starts and removed when it finishes.

## Startup and Shutdown Commands

```python
middleware = ShellToolMiddleware(
    workspace_root="./workspace",
    startup_commands=[
        "python -m venv .venv",
        ". .venv/bin/activate && pip install -r requirements.txt",
    ],
    shutdown_commands="rm -f .session.lock",
)
```

Startup commands must succeed. Shutdown-command failures are logged during cleanup.

## Custom Shell

```python
middleware = ShellToolMiddleware(
    shell_command=("/bin/bash", "--noprofile", "--norc")
)
```

## Custom Environment

```python
middleware = ShellToolMiddleware(
    env={
        "APP_ENV": "development",
        "MAX_WORKERS": 4,
    }
)
```

Environment values are converted to strings.

## Docker Isolation

```python
from langchain.agents.middleware import (
    DockerExecutionPolicy,
    ShellToolMiddleware,
)

middleware = ShellToolMiddleware(
    execution_policy=DockerExecutionPolicy()
)
```

## Redact Command Output

```python
from langchain.agents.middleware import RedactionRule, ShellToolMiddleware

middleware = ShellToolMiddleware(
    redaction_rules=[
        RedactionRule(
            pii_type="email",
            strategy="redact"
        )
    ]
)
```

# Exceptions

The module may raise:

* `ValueError`
  * No shell command arguments were supplied.
  * Shell tool input contains neither `command` nor `restart`.
  * Shell tool input contains both `command` and `restart`.

* `TypeError`
  * An environment-variable name is not a string.

* `RuntimeError`
  * Shell pipes cannot be initialized.
  * The shell session is not running.
  * A startup command fails or times out.

* `ToolException`
  * A restart fails.
  * The internal shell-tool payload does not contain a valid command.

* `PIIDetectionError`
  * A block-style redaction rule detects sensitive output.
  * The tool layer catches this error and returns an error `ToolMessage`.

# Exports

```python
__all__ = [
    "CodexSandboxExecutionPolicy",
    "DockerExecutionPolicy",
    "HostExecutionPolicy",
    "RedactionRule",
    "ShellToolMiddleware",
]
```

`ShellSession`, `ShellToolState`, and `CommandExecutionResult` are defined in the
module but are not included in `__all__`.

# Source

This reference follows the pinned LangChain source:

```text
libs/langchain_v1/langchain/agents/middleware/shell_tool.py
Commit: 42f8f79293cfb7589e5bc1d74a8ae4dfd0bf15e3
```